<a href="https://colab.research.google.com/github/goumze/Simplilearn_Agentic_AI/blob/feature%2Fcollab/RAG_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required libraries
!pip install pymupdf sentence-transformers pandas numpy

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import fitz  # PyMuPDF
import re
import json
import os
from pathlib import Path
import pandas as pd
from typing import List, Dict
from datetime import datetime

# Configuration
PDF_FOLDER = "/content/drive/MyDrive/RAG_Data/pdfs"
OUTPUT_FOLDER = "/content/drive/MyDrive/RAG_Data/chunks"
JSONL_OUTPUT = f"{OUTPUT_FOLDER}/chunks.jsonl"

# Create directories if they don't exist
os.makedirs(PDF_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

def extract_text_from_pdf(pdf_path: str) -> str:
    """Extract text from PDF using PyMuPDF"""
    doc = fitz.open(pdf_path)
    text = ""
    for page_num, page in enumerate(doc):
        text += f"\n--- Page {page_num + 1} ---\n"
        text += page.get_text()
    doc.close()
    return text

def normalize_text(text: str) -> str:
    """Normalize text using regex"""
    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text)
    # Remove multiple newlines
    text = re.sub(r'\n+', '\n', text)
    # Remove special characters but keep punctuation
    text = re.sub(r'[^\w\s\.\,\;\:\!\?\-\(\)\[\]\"\'\n]', '', text)
    # Strip leading/trailing whitespace
    text = text.strip()
    return text

def chunk_by_clauses(text: str, min_chunk_size: int = 100, max_chunk_size: int = 500) -> List[str]:
    """Chunk text by clauses using punctuation markers"""
    # Split by sentence-ending punctuation
    sentences = re.split(r'([.!?]+\s+)', text)

    chunks = []
    current_chunk = ""

    for i in range(0, len(sentences), 2):
        sentence = sentences[i]
        punctuation = sentences[i + 1] if i + 1 < len(sentences) else ""

        # Further split by clause markers (commas, semicolons)
        clauses = re.split(r'([,;:]\s+)', sentence + punctuation)

        for j in range(0, len(clauses), 2):
            clause = clauses[j]
            separator = clauses[j + 1] if j + 1 < len(clauses) else ""
            full_clause = clause + separator

            if len(current_chunk) + len(full_clause) <= max_chunk_size:
                current_chunk += full_clause
            else:
                if len(current_chunk) >= min_chunk_size:
                    chunks.append(current_chunk.strip())
                    current_chunk = full_clause
                else:
                    current_chunk += full_clause

    # Add remaining chunk
    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    return chunks

def process_pdfs_to_chunks(pdf_folder: str) -> List[Dict]:
    """Process all PDFs and create chunks with metadata"""
    all_chunks = []
    chunk_id = 0

    pdf_files = list(Path(pdf_folder).glob("*.pdf"))
    print(f"Found {len(pdf_files)} PDF files")

    for pdf_path in pdf_files:
        print(f"\nProcessing: {pdf_path.name}")

        # Extract text
        raw_text = extract_text_from_pdf(str(pdf_path))

        # Normalize text
        normalized_text = normalize_text(raw_text)

        # Chunk text
        chunks = chunk_by_clauses(normalized_text)

        print(f"  Created {len(chunks)} chunks")

        # Create metadata for each chunk
        for idx, chunk in enumerate(chunks):
            chunk_data = {
                "chunk_id": chunk_id,
                "source_file": pdf_path.name,
                "chunk_index": idx,
                "text": chunk,
                "char_count": len(chunk),
                "timestamp": datetime.now().isoformat()
            }
            all_chunks.append(chunk_data)
            chunk_id += 1

    return all_chunks

def save_to_jsonl(chunks: List[Dict], output_path: str):
    """Save chunks to JSONL format"""
    with open(output_path, 'w', encoding='utf-8') as f:
        for chunk in chunks:
            json.dump(chunk, f, ensure_ascii=False)
            f.write('\n')
    print(f"\nSaved {len(chunks)} chunks to {output_path}")

# Main execution
print("Starting RAG data processing pipeline...")
chunks = process_pdfs_to_chunks(PDF_FOLDER)

if chunks:
    save_to_jsonl(chunks, JSONL_OUTPUT)

    # Display sample
    print("\n=== Sample Chunk ===")
    print(json.dumps(chunks[0], indent=2, ensure_ascii=False))

    # Statistics
    df = pd.DataFrame(chunks)
    print("\n=== Statistics ===")
    print(f"Total chunks: {len(chunks)}")
    print(f"Average chunk size: {df['char_count'].mean():.2f} characters")
    print(f"Chunks per document:\n{df.groupby('source_file').size()}")
else:
    print("No PDFs found or no chunks created. Please add PDF files to:", PDF_FOLDER)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Starting RAG data processing pipeline...
Found 1 PDF files

Processing: manual.pdf
  Created 2631 chunks

Saved 2631 chunks to /content/drive/MyDrive/RAG_Data/chunks/chunks.jsonl

=== Sample Chunk ===
{
  "chunk_id": 0,
  "source_file": "manual.pdf",
  "chunk_index": 0,
  "text": "--- Page 1 --- --- Page 2 --- --- Page 3 --- i Manual for Procurement of Goods Second Edition, 2024 Government of India Ministry of Finance Department of Expenditure --- Page 4 --- --- Page 5 --- FOREWORD (Second Edition, 2024) As part of initiatives to improve transparency, fairness, competition, value for money, and good governance in public procurement, the Department of Expenditure,",
  "char_count": 389,
  "timestamp": "2025-12-20T14:31:10.947833"
}

=== Statistics ===
Total chunks: 2631
Average chunk size: 443.98 characters
Chunks per document:
source_file
manual.pdf    2631
d

In [ ]:
# Install required libraries
!pip install pymupdf sentence-transformers pandas pyarrow faiss-cpu rank-bm25 nltk

import fitz  # PyMuPDF
import re
import os
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List, Dict, Tuple
from datetime import datetime
from sentence_transformers import SentenceTransformer
import faiss
from rank_bm25 import BM25Okapi
import nltk
from google.colab import drive

# Download NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab') # Added to address LookupError: Resource punkt_tab not found
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# Mount Google Drive
drive.mount('/content/drive')

# Configuration
PDF_FOLDER = "/content/drive/MyDrive/RAG_Data/pdfs"
OUTPUT_FOLDER = "/content/drive/MyDrive/RAG_Data/processed"
CHUNKS_PARQUET = f"{OUTPUT_FOLDER}/chunks.parquet"
EMBEDDINGS_PARQUET = f"{OUTPUT_FOLDER}/embeddings.parquet"
FAISS_INDEX_PATH = f"{OUTPUT_FOLDER}/faiss_index.bin"

# Create directories
os.makedirs(PDF_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Initialize embedding model
print("Loading MiniLM model...")
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
EMBEDDING_DIM = 384  # MiniLM dimension

def extract_text_from_pdf(pdf_path: str) -> List[Dict]:
    """Extract text from PDF with page metadata"""
    doc = fitz.open(pdf_path)
    pages_data = []

    for page_num, page in enumerate(doc):
        text = page.get_text()
        if text.strip():
            pages_data.append({
                'page_num': page_num + 1,
                'text': text
            })

    doc.close()
    return pages_data

def normalize_text(text: str) -> str:
    """Advanced text normalization"""
    # Fix common PDF extraction issues
    text = re.sub(r'-\n', '', text)  # Fix hyphenation
    text = re.sub(r'\n+', ' ', text)  # Replace newlines with spaces
    text = re.sub(r'\s+', ' ', text)  # Multiple spaces to single
    text = re.sub(r'[^\w\s\.\,\;\:\!\?\-\(\)\[\]\"\\]', '', text)
    return text.strip()

def split_into_clauses(text: str) -> List[str]:
    """Split text into clauses using punctuation"""
    # Split by major punctuation
    clauses = re.split(r'([.!?];\s+)', text)

    result = []
    for i in range(0, len(clauses) - 1, 2):
        clause = clauses[i] + (clauses[i + 1] if i + 1 < len(clauses) else '')
        if clause.strip():
            result.append(clause.strip())

    # Handle last element if odd number
    if len(clauses) % 2 == 1 and clauses[-1].strip():
        result.append(clauses[-1].strip())

    return result

def create_chunks_from_clauses(clauses: List[str],
                                min_chunk_size: int = 150,
                                max_chunk_size: int = 512,
                                overlap: int = 50) -> List[str]:
    """Create chunks from clauses with overlap"""
    chunks = []
    current_chunk = ""

    for i, clause in enumerate(clauses):
        # Try adding clause to current chunk
        potential_chunk = current_chunk + " " + clause if current_chunk else clause

        if len(potential_chunk) <= max_chunk_size:
            current_chunk = potential_chunk
        else:
            # Save current chunk if it meets minimum size
            if len(current_chunk) >= min_chunk_size:
                chunks.append(current_chunk.strip())

                # Create overlap by including last part of previous chunk
                words = current_chunk.split()
                overlap_text = " ".join(words[-overlap//5:]) if len(words) > overlap//5 else ""
                current_chunk = overlap_text + " " + clause if overlap_text else clause
            else:
                current_chunk = potential_chunk

    # Add final chunk
    if current_chunk.strip() and len(current_chunk) >= min_chunk_size:
        chunks.append(current_chunk.strip())

    return chunks

def process_pdfs_to_chunks(pdf_folder: str) -> pd.DataFrame:
    """Process all PDFs and create chunks DataFrame"""
    all_chunks_data = []
    chunk_id = 0

    pdf_files = list(Path(pdf_folder).glob("*.pdf"))
    print(f"\n{'='*60}")
    print(f"Found {len(pdf_files)} PDF files to process")
    print(f"{'='*60}")

    for pdf_path in pdf_files:
        print(f"\n📄 Processing: {pdf_path.name}")

        try:
            # Extract text by page
            pages_data = extract_text_from_pdf(str(pdf_path))

            for page_data in pages_data:
                # Normalize
                normalized_text = normalize_text(page_data['text'])

                # Split into clauses
                clauses = split_into_clauses(normalized_text)

                # Create chunks
                chunks = create_chunks_from_clauses(clauses)

                print(f"   Page {page_data['page_num']}: {len(chunks)} chunks created")

                # Store with metadata
                for idx, chunk in enumerate(chunks):
                    all_chunks_data.append({
                        'chunk_id': chunk_id,
                        'source_file': pdf_path.name,
                        'page_num': page_data['page_num'],
                        'chunk_index': idx,
                        'text': chunk,
                        'char_count': len(chunk),
                        'word_count': len(chunk.split()),
                        'timestamp': datetime.now().isoformat()
                    })
                    chunk_id += 1

        except Exception as e:
            print(f"   ❌ Error processing {pdf_path.name}: {str(e)}")

    df = pd.DataFrame(all_chunks_data)
    print(f"\n✅ Total chunks created: {len(df)}")
    return df

def generate_embeddings(df: pd.DataFrame, batch_size: int = 32) -> np.ndarray:
    """Generate embeddings using MiniLM"""
    print("\n" + "="*60)
    print("Generating embeddings with MiniLM...")
    print("="*60)

    texts = df['text'].tolist()
    embeddings = embedding_model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    print(f"✅ Generated embeddings with shape: {embeddings.shape}")
    return embeddings

def create_faiss_index(embeddings: np.ndarray) -> faiss.Index:
    """Create FAISS index for vector search"""
    print("\nCreating FAISS index...")

    # Normalize embeddings for cosine similarity
    faiss.normalize_L2(embeddings)

    # Create index
    index = faiss.IndexFlatIP(EMBEDDING_DIM)  # Inner Product for cosine similarity
    index.add(embeddings)

    print(f"✅ FAISS index created with {index.ntotal} vectors")
    return index

def save_artifacts(df: pd.DataFrame, embeddings: np.ndarray, faiss_index: faiss.Index):
    """Save all artifacts to disk"""
    print("\n" + "="*60)
    print("Saving artifacts...")
    print("="*60)

    # Save chunks to Parquet
    df.to_parquet(CHUNKS_PARQUET, index=False)
    print(f"✅ Chunks saved to: {CHUNKS_PARQUET}")

    # Save embeddings to Parquet
    embeddings_df = pd.DataFrame(embeddings)
    embeddings_df['chunk_id'] = df['chunk_id'].values
    embeddings_df.to_parquet(EMBEDDINGS_PARQUET, index=False)
    print(f"✅ Embeddings saved to: {EMBEDDINGS_PARQUET}")

    # Save FAISS index
    faiss.write_index(faiss_index, FAISS_INDEX_PATH)
    print(f"✅ FAISS index saved to: {FAISS_INDEX_PATH}")

# ===================== HYBRID RETRIEVAL =====================

class HybridRetriever:
    def __init__(self, df: pd.DataFrame, embeddings: np.ndarray, faiss_index: faiss.Index):
        self.df = df
        self.embeddings = embeddings
        self.faiss_index = faiss_index
        self.stop_words = set(stopwords.words('english'))

        # Prepare BM25
        print("\nInitializing BM25...")
        tokenized_corpus = [self._preprocess_text(text) for text in df['text'].tolist()]
        self.bm25 = BM25Okapi(tokenized_corpus)
        print("✅ BM25 initialized")

    def _preprocess_text(self, text: str) -> List[str]:
        """Tokenize and remove stopwords"""
        tokens = word_tokenize(text.lower())
        return [token for token in tokens if token.isalnum() and token not in self.stop_words]

    def search_semantic(self, query: str, top_k: int = 10) -> List[Tuple[int, float]]:
        """Semantic search using FAISS"""
        query_embedding = embedding_model.encode([query], convert_to_numpy=True)
        faiss.normalize_L2(query_embedding)

        distances, indices = self.faiss_index.search(query_embedding, top_k)
        return [(int(idx), float(score)) for idx, score in zip(indices[0], distances[0])]

    def search_keyword(self, query: str, top_k: int = 10) -> List[Tuple[int, float]]:
        """Keyword search using BM25"""
        tokenized_query = self._preprocess_text(query)
        scores = self.bm25.get_scores(tokenized_query)

        # Get top k indices
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [(int(idx), float(scores[idx])) for idx in top_indices]

    def hybrid_search(self, query: str, top_k: int = 5,
                     semantic_weight: float = 0.7) -> pd.DataFrame:
        """Hybrid search combining semantic and keyword search"""
        print(f"\n🔍 Query: {query}")
        print("="*60)

        # Get results from both methods
        semantic_results = self.search_semantic(query, top_k=top_k*2)
        keyword_results = self.search_keyword(query, top_k=top_k*2)

        # Normalize scores to 0-1 range
        semantic_scores = {idx: score for idx, score in semantic_results}
        keyword_scores = {idx: score for idx, score in keyword_results}

        # Normalize
        max_sem = max(semantic_scores.values()) if semantic_scores else 1
        max_key = max(keyword_scores.values()) if keyword_scores else 1

        semantic_scores = {k: v/max_sem for k, v in semantic_scores.items()}
        keyword_scores = {k: v/max_key for k, v in keyword_scores.items()}

        # Combine scores
        all_indices = set(semantic_scores.keys()) | set(keyword_scores.keys())
        hybrid_scores = {}

        for idx in all_indices:
            sem_score = semantic_scores.get(idx, 0)
            key_score = keyword_scores.get(idx, 0)
            hybrid_scores[idx] = (semantic_weight * sem_score +
                                 (1 - semantic_weight) * key_score)

        # Sort by hybrid score
        sorted_indices = sorted(hybrid_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

        # Get results
        results = []
        for rank, (idx, score) in enumerate(sorted_indices, 1):
            row = self.df.iloc[idx].to_dict()
            row['rank'] = rank
            row['hybrid_score'] = score
            row['semantic_score'] = semantic_scores.get(idx, 0)
            row['keyword_score'] = keyword_scores.get(idx, 0)
            results.append(row)

        results_df = pd.DataFrame(results)
        return results_df

# ===================== MAIN EXECUTION =====================

def main():
    print("\n" + "="*60)
    print("🚀 RAG SYSTEM - PROCESSING PIPELINE")
    print("="*60)

    # Step 1: Process PDFs
    chunks_df = process_pdfs_to_chunks(PDF_FOLDER)

    if len(chunks_df) == 0:
        print("\n❌ No chunks created. Please add PDF files to:", PDF_FOLDER)
        return

    # Step 2: Generate embeddings
    embeddings = generate_embeddings(chunks_df)

    # Step 3: Create FAISS index
    faiss_index = create_faiss_index(embeddings)

    # Step 4: Save artifacts
    save_artifacts(chunks_df, embeddings, faiss_index)

    # Step 5: Display statistics
    print("\n" + "="*60)
    print("📊 STATISTICS")
    print("="*60)
    print(f"Total chunks: {len(chunks_df)}")
    print(f"Average chunk size: {chunks_df['char_count'].mean():.0f} characters")
    print(f"Average words per chunk: {chunks_df['word_count'].mean():.0f}")
    print(f"\nChunks per document:")
    print(chunks_df.groupby('source_file').size())

    # Display sample
    print("\n" + "="*60)
    print("📝 SAMPLE CHUNK")
    print("="*60)
    sample = chunks_df.iloc[0]
    print(f"Source: {sample['source_file']} (Page {sample['page_num']})")
    print(f"Text: {sample['text'][:200]}...")

    return chunks_df, embeddings, faiss_index

# Run pipeline
chunks_df, embeddings, faiss_index = main()

# ===================== RETRIEVAL DEMO =====================

# Initialize retriever
print("\n" + "="*60)
print("🔧 Initializing Hybrid Retriever...")
print("="*60)
retriever = HybridRetriever(chunks_df, embeddings, faiss_index)

# Example queries
print("\n" + "="*60)
print("💡 RETRIEVAL EXAMPLES")
print("="*60)

# Example 1
results = retriever.hybrid_search(
    "What is machine learning?",
    top_k=3,
    semantic_weight=0.7
)

print("\n📋 Top 3 Results:")
for _, row in results.iterrows():
    print(f"\n🏆 Rank {row['rank']} (Score: {row['hybrid_score']:.3f})")
    print(f"   Source: {row['source_file']} | Page: {row['page_num']}")
    print(f"   Semantic: {row['semantic_score']:.3f} | Keyword: {row['keyword_score']:.3f}")
    print(f"   Text: {row['text'][:150]}...")

# Function to load saved artifacts
def load_artifacts():
    """Load saved artifacts for future use"""
    chunks_df = pd.read_parquet(CHUNKS_PARQUET)
    embeddings_df = pd.read_parquet(EMBEDDINGS_PARQUET)
    embeddings = embeddings_df.drop(columns=['chunk_id']).values
    faiss_index = faiss.read_index(FAISS_INDEX_PATH)

    print("✅ Artifacts loaded successfully")
    return chunks_df, embeddings, faiss_index

print("\n" + "="*60)
print("✅ RAG SYSTEM READY!")
print("="*60)
print("\nUsage:")
print("  results = retriever.hybrid_search('your query', top_k=5)")
print("\nTo reload artifacts later:")
print("  chunks_df, embeddings, faiss_index = load_artifacts()")
print("  retriever = HybridRetriever(chunks_df, embeddings, faiss_index)")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading MiniLM model...

🚀 RAG SYSTEM - PROCESSING PIPELINE

Found 1 PDF files to process

📄 Processing: manual.pdf
   Page 3: 0 chunks created
   Page 5: 1 chunks created
   Page 7: 1 chunks created
   Page 9: 1 chunks created
   Page 10: 1 chunks created
   Page 11: 1 chunks created
   Page 12: 1 chunks created
   Page 13: 1 chunks created
   Page 15: 1 chunks created
   Page 16: 1 chunks created
   Page 17: 1 chunks created
   Page 19: 1 chunks created
   Page 20: 1 chunks created
   Page 21: 2 chunks created
   Page 22: 1 chunks created
   Page 23: 1 chunks created
   Page 24: 1 chunks created
   Page 25: 1 chunks created
   Page 26: 1 chunks created
   Page 27: 1 chunks created
   Page 28: 1 chunks created
   Page 29: 1 chunks created
   Page 30: 1 chunks created
   Page 31: 1 chunks created
   Page 32: 1 chunks created
   Page 33: 1 chunks created
   Pa

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

✅ Generated embeddings with shape: (380, 384)

Creating FAISS index...
✅ FAISS index created with 380 vectors

Saving artifacts...
✅ Chunks saved to: /content/drive/MyDrive/RAG_Data/processed/chunks.parquet
✅ Embeddings saved to: /content/drive/MyDrive/RAG_Data/processed/embeddings.parquet
✅ FAISS index saved to: /content/drive/MyDrive/RAG_Data/processed/faiss_index.bin

📊 STATISTICS
Total chunks: 380
Average chunk size: 3062 characters
Average words per chunk: 477

Chunks per document:
source_file
manual.pdf    380
dtype: int64

📝 SAMPLE CHUNK
Source: manual.pdf (Page 5)
Text: FOREWORD (Second Edition, 2024) As part of initiatives to improve transparency, fairness, competition, value for money, and good governance in public procurement, the Department of Expenditure, Minist...

🔧 Initializing Hybrid Retriever...

Initializing BM25...
✅ BM25 initialized

💡 RETRIEVAL EXAMPLES

🔍 Query: What is machine learning?

📋 Top 3 Results:

🏆 Rank 1 (Score: 0.700)
   Source: manual.pdf | Page: 115

In [ ]:
# Install required libraries
!pip install pymupdf sentence-transformers pandas pyarrow faiss-cpu rank-bm25 nltk

import fitz  # PyMuPDF
import re
import os
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List, Dict, Tuple
from datetime import datetime
from sentence_transformers import SentenceTransformer
import faiss
from rank_bm25 import BM25Okapi
import nltk
from google.colab import drive

# Download NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab') # Added to address LookupError: Resource punkt_tab not found
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# Mount Google Drive
drive.mount('/content/drive')

# Configuration
PDF_FOLDER = "/content/drive/MyDrive/RAG_Data/pdfs"
OUTPUT_FOLDER = "/content/drive/MyDrive/RAG_Data/processed"
CHUNKS_PARQUET = f"{OUTPUT_FOLDER}/chunks.parquet"
EMBEDDINGS_PARQUET = f"{OUTPUT_FOLDER}/embeddings.parquet"
FAISS_INDEX_PATH = f"{OUTPUT_FOLDER}/faiss_index.bin"

# Create directories
os.makedirs(PDF_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Initialize embedding model
print("Loading MiniLM model...")
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
EMBEDDING_DIM = 384  # MiniLM dimension

def extract_text_from_pdf(pdf_path: str) -> List[Dict]:
    """Extract text from PDF with page metadata"""
    doc = fitz.open(pdf_path)
    pages_data = []

    for page_num, page in enumerate(doc):
        text = page.get_text()
        if text.strip():
            pages_data.append({
                'page_num': page_num + 1,
                'text': text
            })

    doc.close()
    return pages_data

def normalize_text(text: str) -> str:
    """Advanced text normalization"""
    # Fix common PDF extraction issues
    text = re.sub(r'-\n', '', text)  # Fix hyphenation
    text = re.sub(r'\n+', ' ', text)  # Replace newlines with spaces
    text = re.sub(r'\s+', ' ', text)  # Multiple spaces to single
    text = re.sub(r'[^\w\s\.\,\;\:\!\?\-\(\)\[\]\"\\]', '', text)
    return text.strip()

def split_into_clauses(text: str) -> List[str]:
    """Split text into clauses using punctuation"""
    # Split by major punctuation
    clauses = re.split(r'([.!?];\s+)', text)

    result = []
    for i in range(0, len(clauses) - 1, 2):
        clause = clauses[i] + (clauses[i + 1] if i + 1 < len(clauses) else '')
        if clause.strip():
            result.append(clause.strip())

    # Handle last element if odd number
    if len(clauses) % 2 == 1 and clauses[-1].strip():
        result.append(clauses[-1].strip())

    return result

def create_chunks_from_clauses(clauses: List[str],
                                min_chunk_size: int = 150,
                                max_chunk_size: int = 512,
                                overlap: int = 50) -> List[str]:
    """Create chunks from clauses with overlap"""
    chunks = []
    current_chunk = ""

    for i, clause in enumerate(clauses):
        # Try adding clause to current chunk
        potential_chunk = current_chunk + " " + clause if current_chunk else clause

        if len(potential_chunk) <= max_chunk_size:
            current_chunk = potential_chunk
        else:
            # Save current chunk if it meets minimum size
            if len(current_chunk) >= min_chunk_size:
                chunks.append(current_chunk.strip())

                # Create overlap by including last part of previous chunk
                words = current_chunk.split()
                overlap_text = " ".join(words[-overlap//5:]) if len(words) > overlap//5 else ""
                current_chunk = overlap_text + " " + clause if overlap_text else clause
            else:
                current_chunk = potential_chunk

    # Add final chunk
    if current_chunk.strip() and len(current_chunk) >= min_chunk_size:
        chunks.append(current_chunk.strip())

    return chunks

def process_pdfs_to_chunks(pdf_folder: str) -> pd.DataFrame:
    """Process all PDFs and create chunks DataFrame"""
    all_chunks_data = []
    chunk_id = 0

    pdf_files = list(Path(pdf_folder).glob("*.pdf"))
    print(f"\n{'='*60}")
    print(f"Found {len(pdf_files)} PDF files to process")
    print(f"{'='*60}")

    for pdf_path in pdf_files:
        print(f"\n📄 Processing: {pdf_path.name}")

        try:
            # Extract text by page
            pages_data = extract_text_from_pdf(str(pdf_path))

            for page_data in pages_data:
                # Normalize
                normalized_text = normalize_text(page_data['text'])

                # Split into clauses
                clauses = split_into_clauses(normalized_text)

                # Create chunks
                chunks = create_chunks_from_clauses(clauses)

                print(f"   Page {page_data['page_num']}: {len(chunks)} chunks created")

                # Store with metadata
                for idx, chunk in enumerate(chunks):
                    all_chunks_data.append({
                        'chunk_id': chunk_id,
                        'source_file': pdf_path.name,
                        'page_num': page_data['page_num'],
                        'chunk_index': idx,
                        'text': chunk,
                        'char_count': len(chunk),
                        'word_count': len(chunk.split()),
                        'timestamp': datetime.now().isoformat()
                    })
                    chunk_id += 1

        except Exception as e:
            print(f"   ❌ Error processing {pdf_path.name}: {str(e)}")

    df = pd.DataFrame(all_chunks_data)
    print(f"\n✅ Total chunks created: {len(df)}")
    return df

def generate_embeddings(df: pd.DataFrame, batch_size: int = 32) -> np.ndarray:
    """Generate embeddings using MiniLM"""
    print("\n" + "="*60)
    print("Generating embeddings with MiniLM...")
    print("="*60)

    texts = df['text'].tolist()
    embeddings = embedding_model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    print(f"✅ Generated embeddings with shape: {embeddings.shape}")
    return embeddings

def create_faiss_index(embeddings: np.ndarray) -> faiss.Index:
    """Create FAISS index for vector search"""
    print("\nCreating FAISS index...")

    # Normalize embeddings for cosine similarity
    faiss.normalize_L2(embeddings)

    # Create index
    index = faiss.IndexFlatIP(EMBEDDING_DIM)  # Inner Product for cosine similarity
    index.add(embeddings)

    print(f"✅ FAISS index created with {index.ntotal} vectors")
    return index

def save_artifacts(df: pd.DataFrame, embeddings: np.ndarray, faiss_index: faiss.Index):
    """Save all artifacts to disk"""
    print("\n" + "="*60)
    print("Saving artifacts...")
    print("="*60)

    # Save chunks to Parquet
    df.to_parquet(CHUNKS_PARQUET, index=False)
    print(f"✅ Chunks saved to: {CHUNKS_PARQUET}")

    # Save embeddings to Parquet
    embeddings_df = pd.DataFrame(embeddings)
    embeddings_df['chunk_id'] = df['chunk_id'].values
    embeddings_df.to_parquet(EMBEDDINGS_PARQUET, index=False)
    print(f"✅ Embeddings saved to: {EMBEDDINGS_PARQUET}")

    # Save FAISS index
    faiss.write_index(faiss_index, FAISS_INDEX_PATH)
    print(f"✅ FAISS index saved to: {FAISS_INDEX_PATH}")

# ===================== HYBRID RETRIEVAL =====================

class HybridRetriever:
    def __init__(self, df: pd.DataFrame, embeddings: np.ndarray, faiss_index: faiss.Index):
        self.df = df
        self.embeddings = embeddings
        self.faiss_index = faiss_index
        self.stop_words = set(stopwords.words('english'))

        # Prepare BM25
        print("\nInitializing BM25...")
        tokenized_corpus = [self._preprocess_text(text) for text in df['text'].tolist()]
        self.bm25 = BM25Okapi(tokenized_corpus)
        print("✅ BM25 initialized")

    def _preprocess_text(self, text: str) -> List[str]:
        """Tokenize and remove stopwords"""
        tokens = word_tokenize(text.lower())
        return [token for token in tokens if token.isalnum() and token not in self.stop_words]

    def search_semantic(self, query: str, top_k: int = 10) -> List[Tuple[int, float]]:
        """Semantic search using FAISS"""
        query_embedding = embedding_model.encode([query], convert_to_numpy=True)
        faiss.normalize_L2(query_embedding)

        distances, indices = self.faiss_index.search(query_embedding, top_k)
        return [(int(idx), float(score)) for idx, score in zip(indices[0], distances[0])]

    def search_keyword(self, query: str, top_k: int = 10) -> List[Tuple[int, float]]:
        """Keyword search using BM25"""
        tokenized_query = self._preprocess_text(query)
        scores = self.bm25.get_scores(tokenized_query)

        # Get top k indices
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [(int(idx), float(scores[idx])) for idx in top_indices]

    def hybrid_search(self, query: str, top_k: int = 5,
                     semantic_weight: float = 0.7) -> pd.DataFrame:
        """Hybrid search combining semantic and keyword search"""
        print(f"\n🔍 Query: {query}")
        print("="*60)

        # Get results from both methods
        semantic_results = self.search_semantic(query, top_k=top_k*2)
        keyword_results = self.search_keyword(query, top_k=top_k*2)

        # Normalize scores to 0-1 range
        semantic_scores = {idx: score for idx, score in semantic_results}
        keyword_scores = {idx: score for idx, score in keyword_results}

        # Normalize
        max_sem = max(semantic_scores.values()) if semantic_scores else 1
        max_key = max(keyword_scores.values()) if keyword_scores else 1

        semantic_scores = {k: v/max_sem for k, v in semantic_scores.items()}
        keyword_scores = {k: v/max_key for k, v in keyword_scores.items()}

        # Combine scores
        all_indices = set(semantic_scores.keys()) | set(keyword_scores.keys())
        hybrid_scores = {}

        for idx in all_indices:
            sem_score = semantic_scores.get(idx, 0)
            key_score = keyword_scores.get(idx, 0)
            hybrid_scores[idx] = (semantic_weight * sem_score +
                                 (1 - semantic_weight) * key_score)

        # Sort by hybrid score
        sorted_indices = sorted(hybrid_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

        # Get results
        results = []
        for rank, (idx, score) in enumerate(sorted_indices, 1):
            row = self.df.iloc[idx].to_dict()
            row['rank'] = rank
            row['hybrid_score'] = score
            row['semantic_score'] = semantic_scores.get(idx, 0)
            row['keyword_score'] = keyword_scores.get(idx, 0)
            results.append(row)

        results_df = pd.DataFrame(results)
        return results_df

# ===================== MAIN EXECUTION =====================

def main():
    print("\n" + "="*60)
    print("🚀 RAG SYSTEM - PROCESSING PIPELINE")
    print("="*60)

    # Step 1: Process PDFs
    chunks_df = process_pdfs_to_chunks(PDF_FOLDER)

    if len(chunks_df) == 0:
        print("\n❌ No chunks created. Please add PDF files to:", PDF_FOLDER)
        return

    # Step 2: Generate embeddings
    embeddings = generate_embeddings(chunks_df)

    # Step 3: Create FAISS index
    faiss_index = create_faiss_index(embeddings)

    # Step 4: Save artifacts
    save_artifacts(chunks_df, embeddings, faiss_index)

    # Step 5: Display statistics
    print("\n" + "="*60)
    print("📊 STATISTICS")
    print("="*60)
    print(f"Total chunks: {len(chunks_df)}")
    print(f"Average chunk size: {chunks_df['char_count'].mean():.0f} characters")
    print(f"Average words per chunk: {chunks_df['word_count'].mean():.0f}")
    print(f"\nChunks per document:")
    print(chunks_df.groupby('source_file').size())

    # Display sample
    print("\n" + "="*60)
    print("📝 SAMPLE CHUNK")
    print("="*60)
    sample = chunks_df.iloc[0]
    print(f"Source: {sample['source_file']} (Page {sample['page_num']})")
    print(f"Text: {sample['text'][:200]}...")

    return chunks_df, embeddings, faiss_index

# ===================== COSINE SIMILARITY UTILITIES =====================

def cosine_similarity(vec1: np.ndarray, vec2: np.ndarray) -> float:
    """Compute cosine similarity between two vectors"""
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

def compute_cosine_similarities(query_embedding: np.ndarray,
                                document_embeddings: np.ndarray) -> np.ndarray:
    """Compute cosine similarities between query and all documents"""
    # Normalize embeddings
    query_norm = query_embedding / np.linalg.norm(query_embedding)
    docs_norm = document_embeddings / np.linalg.norm(document_embeddings, axis=1, keepdims=True)

    # Compute similarities
    similarities = np.dot(docs_norm, query_norm.T).flatten()
    return similarities

def get_top_k_by_cosine(query: str,
                        embeddings: np.ndarray,
                        df: pd.DataFrame,
                        top_k: int = 5,
                        threshold: float = 0.0) -> pd.DataFrame:
    """Retrieve top-k chunks by cosine similarity with optional threshold"""
    print(f"\n🔍 Cosine Similarity Search: {query}")
    print("="*60)

    # Generate query embedding
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)[0]

    # Compute similarities
    similarities = compute_cosine_similarities(query_embedding, embeddings)

    # Apply threshold filter
    mask = similarities >= threshold
    filtered_indices = np.where(mask)[0]
    filtered_similarities = similarities[filtered_indices]

    # Sort by similarity
    sorted_idx = np.argsort(filtered_similarities)[::-1][:top_k]
    top_indices = filtered_indices[sorted_idx]
    top_similarities = filtered_similarities[sorted_idx]

    # Build results
    results = []
    for rank, (idx, sim) in enumerate(zip(top_indices, top_similarities), 1):
        row = df.iloc[idx].to_dict()
        row['rank'] = rank
        row['cosine_similarity'] = float(sim)
        results.append(row)

    results_df = pd.DataFrame(results)

    print(f"✅ Found {len(results_df)} results (threshold: {threshold})")
    return results_df

def compute_similarity_matrix(embeddings: np.ndarray, sample_size: int = 100) -> np.ndarray:
    """Compute pairwise cosine similarity matrix for sample of embeddings"""
    print(f"\nComputing similarity matrix for {sample_size} samples...")

    # Sample embeddings if too large
    if len(embeddings) > sample_size:
        indices = np.random.choice(len(embeddings), sample_size, replace=False)
        sample_embeddings = embeddings[indices]
    else:
        sample_embeddings = embeddings

    # Normalize
    norms = np.linalg.norm(sample_embeddings, axis=1, keepdims=True)
    normalized = sample_embeddings / norms

    # Compute similarity matrix
    similarity_matrix = np.dot(normalized, normalized.T)

    print(f"✅ Similarity matrix shape: {similarity_matrix.shape}")
    return similarity_matrix

def find_similar_chunks(chunk_id: int,
                        embeddings: np.ndarray,
                        df: pd.DataFrame,
                        top_k: int = 5) -> pd.DataFrame:
    """Find most similar chunks to a given chunk"""
    if chunk_id >= len(embeddings):
        raise ValueError(f"Chunk ID {chunk_id} out of range")

    chunk_embedding = embeddings[chunk_id]
    similarities = compute_cosine_similarities(chunk_embedding, embeddings)

    # Exclude the query chunk itself
    similarities[chunk_id] = -1

    # Get top k
    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []
    for rank, idx in enumerate(top_indices, 1):
        row = df.iloc[idx].to_dict()
        row['rank'] = rank
        row['cosine_similarity'] = float(similarities[idx])
        results.append(row)

    return pd.DataFrame(results)

def analyze_similarity_distribution(embeddings: np.ndarray,
                                   num_samples: int = 1000) -> Dict:
    """Analyze the distribution of cosine similarities in the corpus"""
    print("\n📊 Analyzing similarity distribution...")

    # Random sample pairs
    n = len(embeddings)
    indices1 = np.random.choice(n, num_samples, replace=True)
    indices2 = np.random.choice(n, num_samples, replace=True)

    similarities = []
    for i1, i2 in zip(indices1, indices2):
        if i1 != i2:
            sim = cosine_similarity(embeddings[i1], embeddings[i2])
            similarities.append(sim)

    similarities = np.array(similarities)

    stats = {
        'mean': float(np.mean(similarities)),
        'std': float(np.std(similarities)),
        'min': float(np.min(similarities)),
        'max': float(np.max(similarities)),
        'median': float(np.median(similarities)),
        'q25': float(np.percentile(similarities, 25)),
        'q75': float(np.percentile(similarities, 75))
    }

    print("\n📈 Similarity Statistics:")
    for key, value in stats.items():
        print(f"   {key:8s}: {value:.4f}")

    return stats



# Run pipeline
chunks_df, embeddings, faiss_index = main()

# ===================== COSINE SIMILARITY DEMOS =====================

print("\n" + "="*60)
print("🎯 COSINE SIMILARITY ANALYSIS")
print("="*60)

# Demo 1: Direct cosine similarity search
print("\n1️⃣ Direct Cosine Similarity Search")
cosine_results = get_top_k_by_cosine(
    query="machine learning algorithms",
    embeddings=embeddings,
    df=chunks_df,
    top_k=3,
    threshold=0.3  # Only show results with similarity >= 0.3
)

for _, row in cosine_results.iterrows():
    print(f"\n🏆 Rank {row['rank']} (Similarity: {row['cosine_similarity']:.4f})")
    print(f"   Source: {row['source_file']} | Page: {row['page_num']}")
    print(f"   Text: {row['text'][:150]}...")

# Demo 2: Find similar chunks
print("\n\n2️⃣ Finding Similar Chunks to Chunk ID 0")
similar_chunks = find_similar_chunks(
    chunk_id=0,
    embeddings=embeddings,
    df=chunks_df,
    top_k=3
)

print(f"\n📄 Original Chunk: {chunks_df.iloc[0]['text'][:100]}...")
print("\n🔗 Most Similar Chunks:")
for _, row in similar_chunks.iterrows():
    print(f"\n   Rank {row['rank']} (Similarity: {row['cosine_similarity']:.4f})")
    print(f"   {row['text'][:100]}...")

# Demo 3: Similarity distribution
similarity_stats = analyze_similarity_distribution(embeddings, num_samples=1000)

# Demo 4: Compare with hybrid search
print("\n\n3️⃣ Comparison: Cosine vs Hybrid Search")
query = "artificial intelligence"

print(f"\n🔵 Cosine Similarity Only:")
cosine_only = get_top_k_by_cosine(query, embeddings, chunks_df, top_k=3)
for _, row in cosine_only.iterrows():
    print(f"   Rank {row['rank']}: Score={row['cosine_similarity']:.4f}")
    print(f"   {row['text'][:80]}...")

print(f"\n🟣 Hybrid Search (Cosine + BM25):")
hybrid_results = retriever.hybrid_search(query, top_k=3, semantic_weight=0.7)
for _, row in hybrid_results.iterrows():
    print(f"   Rank {row['rank']}: Hybrid={row['hybrid_score']:.4f}, "
          f"Semantic={row['semantic_score']:.4f}, Keyword={row['keyword_score']:.4f}")
    print(f"   {row['text'][:80]}...")

# Helper function for custom similarity search
def similarity_search(query: str, top_k: int = 5, threshold: float = 0.0):
    """Convenient wrapper for cosine similarity search"""
    return get_top_k_by_cosine(query, embeddings, chunks_df, top_k, threshold)

print("\n" + "="*60)
print("✅ COSINE SIMILARITY TOOLS READY!")
print("="*60)
print("\nQuick Usage:")
print("  # Search by similarity")
print("  results = similarity_search('your query', top_k=5, threshold=0.3)")
print("\n  # Find similar chunks")
print("  similar = find_similar_chunks(chunk_id=0, embeddings, chunks_df, top_k=5)")
print("\n  # Analyze corpus")
print("  stats = analyze_similarity_distribution(embeddings)")


# ===================== RETRIEVAL DEMO =====================

# Initialize retriever
print("\n" + "="*60)
print("🔧 Initializing Hybrid Retriever...")
print("="*60)
retriever = HybridRetriever(chunks_df, embeddings, faiss_index)

# Example queries
print("\n" + "="*60)
print("💡 RETRIEVAL EXAMPLES")
print("="*60)

# Example 1
results = retriever.hybrid_search(
    "What is machine learning?",
    top_k=3,
    semantic_weight=0.7
)

print("\n📋 Top 3 Results:")
for _, row in results.iterrows():
    print(f"\n🏆 Rank {row['rank']} (Score: {row['hybrid_score']:.3f})")
    print(f"   Source: {row['source_file']} | Page: {row['page_num']}")
    print(f"   Semantic: {row['semantic_score']:.3f} | Keyword: {row['keyword_score']:.3f}")
    print(f"   Text: {row['text'][:150]}...")

# Function to load saved artifacts
def load_artifacts():
    """Load saved artifacts for future use"""
    chunks_df = pd.read_parquet(CHUNKS_PARQUET)
    embeddings_df = pd.read_parquet(EMBEDDINGS_PARQUET)
    embeddings = embeddings_df.drop(columns=['chunk_id']).values
    faiss_index = faiss.read_index(FAISS_INDEX_PATH)

    print("✅ Artifacts loaded successfully")
    return chunks_df, embeddings, faiss_index

print("\n" + "="*60)
print("✅ RAG SYSTEM READY!")
print("="*60)
print("\nUsage:")
print("  results = retriever.hybrid_search('your query', top_k=5)")
print("\nTo reload artifacts later:")
print("  chunks_df, embeddings, faiss_index = load_artifacts()")
print("  retriever = HybridRetriever(chunks_df, embeddings, faiss_index)")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading MiniLM model...

🚀 RAG SYSTEM - PROCESSING PIPELINE

Found 1 PDF files to process

📄 Processing: manual.pdf
   Page 3: 0 chunks created
   Page 5: 1 chunks created
   Page 7: 1 chunks created
   Page 9: 1 chunks created
   Page 10: 1 chunks created
   Page 11: 1 chunks created
   Page 12: 1 chunks created
   Page 13: 1 chunks created
   Page 15: 1 chunks created
   Page 16: 1 chunks created
   Page 17: 1 chunks created
   Page 19: 1 chunks created
   Page 20: 1 chunks created
   Page 21: 2 chunks created
   Page 22: 1 chunks created
   Page 23: 1 chunks created
   Page 24: 1 chunks created
   Page 25: 1 chunks created
   Page 26: 1 chunks created
   Page 27: 1 chunks created
   Page 28: 1 chunks created
   Page 29: 1 chunks created
   Page 30: 1 chunks created
   Page 31: 1 chunks created
   Page 32: 1 chunks created
   Page 33: 1 chunks created
   Pa

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

✅ Generated embeddings with shape: (380, 384)

Creating FAISS index...
✅ FAISS index created with 380 vectors

Saving artifacts...
✅ Chunks saved to: /content/drive/MyDrive/RAG_Data/processed/chunks.parquet
✅ Embeddings saved to: /content/drive/MyDrive/RAG_Data/processed/embeddings.parquet
✅ FAISS index saved to: /content/drive/MyDrive/RAG_Data/processed/faiss_index.bin

📊 STATISTICS
Total chunks: 380
Average chunk size: 3062 characters
Average words per chunk: 477

Chunks per document:
source_file
manual.pdf    380
dtype: int64

📝 SAMPLE CHUNK
Source: manual.pdf (Page 5)
Text: FOREWORD (Second Edition, 2024) As part of initiatives to improve transparency, fairness, competition, value for money, and good governance in public procurement, the Department of Expenditure, Minist...

🎯 COSINE SIMILARITY ANALYSIS

1️⃣ Direct Cosine Similarity Search

🔍 Cosine Similarity Search: machine learning algorithms
✅ Found 0 results (threshold: 0.3)


2️⃣ Finding Similar Chunks to Chunk ID 0

📄 Origin